# Dataset plots

## Part A — Review of every recording

In [ ]:
from itertools import groupby
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from src import plots as plots_b
from src.analyze import analyze_file, run_analysis
from src.blink_signal import has_blink_columns
from src.detection_config import DetectionConfig, ScenarioConfig
from src.gaze_signal import has_gaze_columns
from src.ground_truth import build_expected_blinks, build_expected_saccades
from src.io_utils import collect_csv_files
from src.plots import (
    plot_blink_signal,
    plot_detected_events,
    plot_gaze_speed,
    plot_saccade_directions,
    plot_speed_histogram,
    plot_threshold_sweep,
)
from src.reports import build_saccade_evaluation

In [ ]:
DATA_DIR = Path("data")
detection_cfg = DetectionConfig()
files = collect_csv_files(DATA_DIR)

In [ ]:
def plot_recording(samples, cfg):
    if has_gaze_columns(samples):
        plot_gaze_speed(samples, cfg)
    if has_blink_columns(samples):
        plot_blink_signal(samples, cfg)

In [ ]:
for participant, participant_files in groupby(files, key=lambda path: path.parent.name):
    display(Markdown(f"## {participant.capitalize()}"))
    for path in participant_files:
        recording_samples, _ = analyze_file(path, detection_cfg)
        plot_recording(recording_samples, detection_cfg)
        plt.show()

## Part B — Thesis figures

In [ ]:
SACCADE_FILE = Path("data/participant_4/SaccadeData_20260729_110441_20s.csv")
BLINK_FILE = Path("data/participant_4/BlinkData_20260729_110615_20s.csv")
FIXATION_FILE = Path("data/participant_4/FixationData_20260729_110316_20s.csv")

thesis_detection_cfg = DetectionConfig()
thesis_scenario_cfg = ScenarioConfig()
thesis_result = run_analysis(thesis_detection_cfg, thesis_scenario_cfg)
all_samples = pd.concat(thesis_result.samples.values(), ignore_index=True)
saccade_samples, saccade_events = analyze_file(SACCADE_FILE, thesis_detection_cfg)
blink_samples, _ = analyze_file(BLINK_FILE, thesis_detection_cfg)
fixation_samples, fixation_events = analyze_file(FIXATION_FILE, thesis_detection_cfg)
expected_saccades = build_expected_saccades(saccade_samples)
expected_blinks = build_expected_blinks(blink_samples, thesis_scenario_cfg)

### P1 — Angular velocity histogram

In [ ]:
plot_speed_histogram(all_samples, thesis_detection_cfg);

### P2 — Detected events and target jumps

In [ ]:
plot_detected_events(
    saccade_samples,
    saccade_events,
    thesis_detection_cfg,
    expected_saccades["expected_time_s"],
);

### P3 — Blink signal and metronome markers

In [ ]:
plots_b.plot_blink_signal(
    blink_samples, thesis_detection_cfg, expected_blinks["expected_time_s"]
);

### P4 — Fixation recording timeline

In [ ]:
plot_detected_events(fixation_samples, fixation_events, thesis_detection_cfg);

### P5 — Saccade direction scatter

In [ ]:
plot_saccade_directions(saccade_events);

### P6 — Velocity threshold sweep

In [ ]:
sweep_rows = []
for threshold in range(40, 181, 20):
    sweep_result = run_analysis(
        DetectionConfig(velocity_threshold_deg_s=float(threshold)),
        thesis_scenario_cfg,
    )
    total = build_saccade_evaluation(
        sweep_result.saccade_matches, sweep_result.events
    ).iloc[-1]
    sweep_rows.append(
        {
            "velocity_threshold_deg_s": threshold,
            "recall": total["recall"],
            "precision": total["precision"],
        }
    )

plot_threshold_sweep(pd.DataFrame(sweep_rows));